# 07 — Refined construct dictionary

Combine all audits into topic→construct and topic×position→construct weights.
**Freeze the dictionary here before looking at refined rating results.**

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

cwd = Path.cwd().resolve()
root = cwd
for _ in range(6):
    if (root / "configs").is_dir() and (root / "src").is_dir():
        break
    root = root.parent
sys.path.insert(0, str(root))

from src.stage11_refined_construct_analysis.analysis import notebook_helpers as nh
from src.stage11_refined_construct_analysis.analysis.constructs import (
    CODE_TO_RAX,
    COMPOSITE_DEFS,
    LOG_RATIO_DEFS,
    normalize_code,
)

ctx = nh.setup("07_refined_construct_dictionary")
cfg = ctx.cfg

Project root : /home/polina/Documents/Cursor_Projects/romantic_novels_large_corpus
Config       : configs/stage11/refined_constructs.yaml
Run          : v4_l12_granular_final_call49
Outputs      : results/stage11_refined_construct_analysis/v4_l12_granular_final_call49/notebook_analysis/07_refined_construct_dictionary


In [2]:
master = nh.load_master(cfg)
freeze = nh.load_freeze(cfg)
print(json.dumps(freeze, indent=2))
assert freeze.get("frozen") is True, "Dictionary must be frozen via 07_build_master_table.py"

dominance = float(cfg.section("weights", "strict_dominance"))
print(f"Strict dominance threshold: {dominance}")
print(f"Weight design: {freeze.get('weight_design', 'legacy')}")
cov = freeze.get("construct_coverage") or nh.load_construct_coverage(cfg)
if cov:
    ratios = pd.DataFrame(
        [
            {"ratio": k, **{kk: vv for kk, vv in v.items() if kk != "topic_ids"}}
            for k, v in (cov.get("ratios") or {}).items()
        ]
    )
    if not ratios.empty:
        display(ratios)
        ctx.save_table(ratios, "construct_ratio_gates")

{
  "run_id": "v4_l12_granular_final_call49",
  "n_topics": 348,
  "n_audited": 157,
  "n_manual_review": 48,
  "code_coverage": {
    "intimacy_code": 98,
    "hea_code": 10,
    "security_code": 90,
    "care_protection_code": 71,
    "darkness_code": 22,
    "arc_role": 54
  },
  "construct_coverage": {
    "atoms": {
      "RAX_appearance_grooming": {
        "n_topics": 6,
        "gate": "viable",
        "topic_ids": [
          18,
          77,
          171,
          218,
          253,
          364
        ]
      },
      "RAX_arc_falling": {
        "n_topics": 8,
        "gate": "viable",
        "topic_ids": [
          96,
          129,
          214,
          265,
          272,
          285,
          293,
          303
        ]
      },
      "RAX_arc_rising": {
        "n_topics": 18,
        "gate": "viable",
        "topic_ids": [
          3,
          29,
          45,
          46,
          56,
          85,
          124,
          126,
          128,
 

,ratio,gate,numerator_topics,denominator_topics
0,RLR_emotional_vs_explicit,viable,33,3
1,RLR_emotional_vs_material_security,unmeasurable,14,0
2,RLR_protection_vs_control,thin,1,3
3,RLR_darkness_vs_tenderness,viable,20,25


  saved table: results/stage11_refined_construct_analysis/v4_l12_granular_final_call49/notebook_analysis/07_refined_construct_dictionary/tables/construct_ratio_gates.csv  (4 rows)


## Weight matrices

In [3]:
for mode in ("strict", "weighted", "inclusive"):
    w = nh.load_weights(cfg, mode)
    w = w.copy()
    w["code_norm"] = w["construct_code"].map(normalize_code)
    print(f"\nW_tk_{mode}: {len(w)} rows, {w['topic_id'].nunique()} topics")
    display(w.groupby("construct_family")["construct_code"].nunique().rename("n_codes").to_frame())
    ctx.save_table(w, f"topic_construct_weights_{mode}")

wtkr = nh.load_w_tkr(cfg)
if not wtkr.empty:
    ctx.save_table(wtkr, "topic_construct_weights_tkr")


W_tk_strict: 172 rows, 106 topics


,n_codes
construct_family,
arc,8
care_protection,7
darkness,2
darkness_bridge,1
hea,2
intimacy,9
security,9
tenderness_bridge,1


  saved table: results/stage11_refined_construct_analysis/v4_l12_granular_final_call49/notebook_analysis/07_refined_construct_dictionary/tables/topic_construct_weights_strict.csv  (172 rows)

W_tk_weighted: 496 rows, 167 topics


,n_codes
construct_family,
arc,9
care_protection,10
darkness,5
darkness_bridge,1
hea,5
intimacy,10
security,14
tenderness_bridge,1


  saved table: results/stage11_refined_construct_analysis/v4_l12_granular_final_call49/notebook_analysis/07_refined_construct_dictionary/tables/topic_construct_weights_weighted.csv  (496 rows)

W_tk_inclusive: 367 rows, 164 topics


,n_codes
construct_family,
arc,9
care_protection,9
darkness,4
darkness_bridge,1
hea,5
intimacy,10
security,14
tenderness_bridge,1


  saved table: results/stage11_refined_construct_analysis/v4_l12_granular_final_call49/notebook_analysis/07_refined_construct_dictionary/tables/topic_construct_weights_inclusive.csv  (367 rows)
  saved table: results/stage11_refined_construct_analysis/v4_l12_granular_final_call49/notebook_analysis/07_refined_construct_dictionary/tables/topic_construct_weights_tkr.csv  (266 rows)


## Code → RAX map (frozen)

In [4]:
rax_map = (
    pd.DataFrame(
        [{"code": k, "rax": v} for k, vs in CODE_TO_RAX.items() for v in vs]
    )
    .sort_values(["rax", "code"])
    .reset_index(drop=True)
)
display(rax_map.head(40))
ctx.save_table(rax_map, "refined_construct_dictionary")

comp = pd.DataFrame(
    [{"composite": k, "parts": ", ".join(v)} for k, v in COMPOSITE_DEFS.items()]
)
ratios = pd.DataFrame(
    [{"name": k, "numerator": v[0], "denominator": v[1]} for k, v in LOG_RATIO_DEFS.items()]
)
display(comp)
display(ratios)
ctx.save_table(comp, "composite_defs")
ctx.save_table(ratios, "log_ratio_defs")

,code,rax
0,S13,RAX_appearance_grooming
1,ARC_1,RAX_arc_falling
2,ARC_2,RAX_arc_falling
3,ARC_3,RAX_arc_falling
4,ARC_4,RAX_arc_falling
5,ARC_5,RAX_arc_rising
6,ARC_6,RAX_arc_rising
7,ARC_7,RAX_arc_rising
8,ARC_8,RAX_arc_rising
9,H4_10,RAX_coercive_control


  saved table: results/stage11_refined_construct_analysis/v4_l12_granular_final_call49/notebook_analysis/07_refined_construct_dictionary/tables/refined_construct_dictionary.csv  (59 rows)


,composite,parts
0,RAX_h1_emotional_side,"RAX_emotional_intimacy, RAX_emotional_reassura..."
1,RAX_h1_explicit_side,RAX_explicit_sex
2,RAX_h2_strict,RAX_final_relational_payoff
3,RAX_h2_broad,"RAX_repair, RAX_mutual_commitment, RAX_final_r..."
4,RAX_h3_emotional_side,"RAX_emotional_security, RAX_commitment_security"
5,RAX_h3_material_side,"RAX_material_provision, RAX_housing_security"
6,RAX_social_presentation,"RAX_status_display, RAX_appearance_grooming, R..."
7,RAX_h4_protection_side,RAX_external_protection
8,RAX_protective_care_broad,"RAX_external_protection, RAX_protective_commit..."
9,RAX_h4_possession_side,"RAX_possessive_claiming, RAX_coercive_control"


,name,numerator,denominator
0,RLR_emotional_vs_explicit,RAX_h1_emotional_side,RAX_h1_explicit_side
1,RLR_emotional_vs_material_security,RAX_h3_emotional_side,RAX_h3_material_side
2,RLR_protection_vs_control,RAX_h4_protection_side,RAX_h4_possession_side
3,RLR_darkness_vs_tenderness,RAX_h5_relational_darkness_side,RAX_h5_tenderness_side


  saved table: results/stage11_refined_construct_analysis/v4_l12_granular_final_call49/notebook_analysis/07_refined_construct_dictionary/tables/composite_defs.csv  (12 rows)
  saved table: results/stage11_refined_construct_analysis/v4_l12_granular_final_call49/notebook_analysis/07_refined_construct_dictionary/tables/log_ratio_defs.csv  (4 rows)


In [5]:
# Coverage after normalisation
cov = []
for col, family in [
    ("intimacy_code", "H1"),
    ("hea_code", "H2"),
    ("security_code", "H3"),
    ("care_protection_code", "H4"),
    ("darkness_code", "H5"),
    ("arc_role", "H6"),
]:
    raw = master[col].notna().sum()
    normed = master[col].map(normalize_code).notna().sum()
    cov.append({"hypothesis": family, "raw_coded": int(raw), "normalised_mapped": int(normed)})
cov_df = pd.DataFrame(cov)
display(cov_df)
ctx.save_table(cov_df, "code_normalisation_coverage")

print(
    "\nDictionary frozen. Do not peek at refined rating effects until notebooks 08–09.\n"
    "Stage 09 taxonomy remains descriptive; this table is the hypothesis measurement layer.\n"
    "Note: RAX_relational_darkness is interpersonal/conflict darkness (7.2/4.4 anchors + D1/D2); "
    "partner-vs-external source for 7.2 was not topic-audited. "
    "H4_5a → RAX_protective_commitment; primary H4 contrast uses external protection only."
)

,hypothesis,raw_coded,normalised_mapped
0,H1,98,89
1,H2,10,10
2,H3,90,89
3,H4,71,71
4,H5,22,21
5,H6,54,49


  saved table: results/stage11_refined_construct_analysis/v4_l12_granular_final_call49/notebook_analysis/07_refined_construct_dictionary/tables/code_normalisation_coverage.csv  (6 rows)

Dictionary frozen. Do not peek at refined rating effects until notebooks 08–09.
Stage 09 taxonomy remains descriptive; this table is the hypothesis measurement layer.
Note: RAX_relational_darkness is interpersonal/conflict darkness (7.2/4.4 anchors + D1/D2); partner-vs-external source for 7.2 was not topic-audited. H4_5a → RAX_protective_commitment; primary H4 contrast uses external protection only.
